In [ ]:
# CELL A: Paths & globals
from pathlib import Path
import os, sys

WORKDIR = Path('/content/research-assistant')
PAPERS_DIR = WORKDIR / 'papers'
PARSED_DIR = WORKDIR / 'parsed'
CHROMA_DIR = WORKDIR / 'chroma_db'

# create folders
for p in [WORKDIR, PAPERS_DIR, PARSED_DIR, CHROMA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("WORKDIR:", WORKDIR)
print("PAPERS_DIR:", PAPERS_DIR)
print("PARSED_DIR:", PARSED_DIR)
print("CHROMA_DIR:", CHROMA_DIR)


WORKDIR: /content/research-assistant
PAPERS_DIR: /content/research-assistant/papers
PARSED_DIR: /content/research-assistant/parsed
CHROMA_DIR: /content/research-assistant/chroma_db


In [ ]:
# CELL B (bash)
%%bash
apt-get update -qq
apt-get install -y -qq poppler-utils tesseract-ocr libgl1-mesa-glx

pip install -q "transformers>=4.35.0" \
                "sentence-transformers>=2.2.2" \
                "chromadb[duckdb]" \
                "pymupdf" \
                "pdfplumber" \
                "pytesseract" \
                "accelerate" \
                "bitsandbytes" \
                "torch" \
                "scikit-learn" \
                "networkx" \
                "nltk" \
                "umap-learn" \
                "plotly" \
                "pdf2image" \
                "tqdm"


Selecting previously unselected package poppler-utils.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 131.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-exporter-otlp-proto-common==1.37.0, but you have opentelemetry-exporter-otlp-proto-common 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-proto==1.37.0, but you have opentelemetry-proto 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-sdk~=1.37.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.
google-a

In [ ]:
# CELL C
import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
# CELL D: Reset Chroma DB folder
import shutil, os, stat
CHROMA_DIR = "/content/research-assistant/chroma_db"

# remove and recreate
shutil.rmtree(CHROMA_DIR, ignore_errors=True)
os.makedirs(CHROMA_DIR, exist_ok=True)

# set permissive permissions
os.chmod(CHROMA_DIR, 0o777)

# sanity test file create
test_path = os.path.join(CHROMA_DIR, "test_write.txt")
with open(test_path, "w") as f:
    f.write("ok")
print("Test write file saved:", test_path)
print("Chroma folder recreated and permissions set to 777")


Test write file saved: /content/research-assistant/chroma_db/test_write.txt
Chroma folder recreated and permissions set to 777


In [ ]:
# CELL E: PDF -> JSON parsing (run after PDFs are in PAPERS_DIR)
import fitz, pdfplumber, pytesseract
from pdf2image import convert_from_path
import re, json
from pathlib import Path
from nltk.tokenize import sent_tokenize

PAPERS_DIR = Path('/content/research-assistant/papers')
PARSED_DIR = Path('/content/research-assistant/parsed')
PARSED_DIR.mkdir(parents=True, exist_ok=True)

def pdf_to_text_mupdf(pdf_path: str) -> str:
    try:
        doc = fitz.open(pdf_path)
        return "\n".join([p.get_text("text") or "" for p in doc])
    except Exception:
        return ""

def pdf_to_text_pdfplumber(pdf_path: str) -> str:
    try:
        with pdfplumber.open(pdf_path) as pdf:
            return "\n".join([p.extract_text() or "" for p in pdf.pages])
    except Exception:
        return ""

def ocr_pdf(pdf_path: str, dpi=200) -> str:
    try:
        images = convert_from_path(pdf_path, dpi=dpi)
        return "\n".join([pytesseract.image_to_string(img) for img in images])
    except Exception:
        return ""

def extract_metadata_from_text(text: str):
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    title = lines[0] if lines else ""
    authors=[]
    if len(lines)>1:
        cleaned = re.sub(r"\S+@\S+", "", lines[1])
        authors = [a.strip() for a in re.split(r",| and |;", cleaned) if a.strip()]
    return {"title": title, "authors": authors}

def split_sections(text: str):
    headings = ["abstract","introduction","related work","background","methods","experiments","results","discussion","conclusion","references"]
    lines = text.splitlines()
    sections={}
    current="full_text"
    buf=[]
    for ln in lines:
        if any(re.match(rf"^\s*{h}\s*$", ln.strip(), flags=re.I) for h in headings):
            sections[current]="\n".join(buf).strip()
            buf=[]
            current=ln.strip().lower()
        else:
            buf.append(ln)
    sections[current] = "\n".join(buf).strip()
    return sections

def ingest_pdf_to_json(pdf_path: Path):
    path = str(pdf_path)
    txt = pdf_to_text_mupdf(path)
    if len(txt.strip()) < 100:
        txt = pdf_to_text_pdfplumber(path)
    if len(txt.strip()) < 100:
        txt = ocr_pdf(path)
    meta = extract_metadata_from_text(txt)
    sections = split_sections(txt)
    out_file = PARSED_DIR / (pdf_path.stem + ".json")
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump({"file": path, "meta": meta, "sections": sections, "raw_text": txt}, f, indent=2, ensure_ascii=False)
    print("Saved:", out_file)

# Run for all PDFs
pdfs = sorted(PAPERS_DIR.glob("*.pdf"))
print("Found PDFs:", [p.name for p in pdfs])
for pdf in pdfs:
    ingest_pdf_to_json(pdf)
print("Parsing complete. JSON files in:", PARSED_DIR)


Found PDFs: ['1803.06161v2.pdf', '2204.02809v2.pdf', '2212.07286v2.pdf']
Saved: /content/research-assistant/parsed/1803.06161v2.json
Saved: /content/research-assistant/parsed/2204.02809v2.json
Saved: /content/research-assistant/parsed/2212.07286v2.json
Parsing complete. JSON files in: /content/research-assistant/parsed


In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# ==========================================================
# CELL F: Section-Aware Indexing + Improved Chunking
# ==========================================================

from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient
import json
from tqdm.auto import tqdm
from pathlib import Path
from nltk.tokenize import sent_tokenize

# -------------------- PATHS --------------------
PARSED_DIR = Path('/content/research-assistant/parsed')
CHROMA_DIR = "/content/research-assistant/chroma_db"

# ------------------ EMBEDDINGS -----------------
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL, device='cuda')  # use 'cpu' if needed

# ------------------- CHROMA --------------------
client = PersistentClient(path=CHROMA_DIR)
collection = client.get_or_create_collection(name="papers_collection")

# ----------------- PARAMETERS -----------------
CHUNK_SIZE = 800        # words
CHUNK_OVERLAP = 100     # words
BATCH_SIZE = 64

# ----------------- UTILITIES ------------------
def chunk_sentences(sentences, chunk_size=800, overlap=100):
    chunks = []
    current = []
    current_len = 0

    for sent in sentences:
        words = sent.split()
        wlen = len(words)

        if current_len + wlen > chunk_size:
            chunks.append(" ".join(current))

            # overlap
            overlap_words = " ".join(current).split()[-overlap:]
            current = [" ".join(overlap_words)]
            current_len = len(overlap_words)

        current.append(sent)
        current_len += wlen

    if current:
        chunks.append(" ".join(current))

    return chunks


# ------------------ INDEXING -------------------
json_files = sorted(PARSED_DIR.glob("*.json"))
print("Found JSON files:", [j.name for j in json_files])

total_added = 0

for jf in tqdm(json_files, desc="Indexing papers"):
    with open(jf, "r", encoding="utf-8") as f:
        data = json.load(f)

    paper_id = jf.stem
    sections = data.get("sections", {})

    for section_name, section_text in sections.items():
        if not section_text or len(section_text.strip()) < 200:
            continue

        sentences = sent_tokenize(section_text)
        chunks = chunk_sentences(sentences, CHUNK_SIZE, CHUNK_OVERLAP)

        ids = [
            f"{paper_id}__{section_name}__{i:04d}"
            for i in range(len(chunks))
        ]

        metas = [
            {
                "paper_id": paper_id,
                "section": section_name,
                "chunk_id": i
            }
            for i in range(len(chunks))
        ]

        for i in range(0, len(chunks), BATCH_SIZE):
            doc_batch = chunks[i:i + BATCH_SIZE]
            id_batch = ids[i:i + BATCH_SIZE]
            meta_batch = metas[i:i + BATCH_SIZE]

            embeddings = embedder.encode(
                doc_batch,
                convert_to_numpy=True,
                show_progress_bar=False
            )

            collection.add(
                ids=id_batch,
                documents=doc_batch,
                embeddings=embeddings.tolist(),
                metadatas=meta_batch
            )

            total_added += len(doc_batch)

print("\nIndexing complete.")
print("Total chunks added:", total_added)
print("Collection count():", collection.count())


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Found JSON files: ['1803.06161v2.json', '2204.02809v2.json', '2212.07286v2.json']


Indexing papers:   0%|          | 0/3 [00:00<?, ?it/s]


Indexing complete.
Total chunks added: 48
Collection count(): 48


In [ ]:
# ==========================================================
# CELL G: Reranker + Generator (FREE, NON-GATED)
# ==========================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from sentence_transformers import CrossEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ---------------- RERANKER ----------------
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
print("Loading reranker:", RERANKER_MODEL)
reranker = CrossEncoder(RERANKER_MODEL, device=device)
print("Reranker loaded.")

# ---------------- GENERATOR ----------------
GEN_MODEL = "Qwen/Qwen2.5-3B-Instruct"
print("Loading generator:", GEN_MODEL)

bnb = BitsAndBytesConfig(load_in_4bit=True)

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL, use_fast=True)

gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL,
    device_map="auto",
    quantization_config=bnb,
    torch_dtype=torch.float16
)

generator = pipeline(
    "text-generation",
    model=gen_model,
    tokenizer=gen_tokenizer,
    max_new_tokens=450,
    temperature=0,
    do_sample=False
)

print("Generator ready:", GEN_MODEL)


Device: cuda
Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generator ready: Qwen/Qwen2.5-3B-Instruct


In [ ]:
# ==========================================================
# CELL H — Production Citation-Based RAG (NO HALLUCINATION)
# ==========================================================

import numpy as np
from typing import List, Dict


def detect_mode(query: str) -> str:
    q = query.lower().strip()

    # Direct question-answering
    if q.startswith(("what", "why", "how", "which", "does", "is", "are")):
        return "qa"

    # Summary requests
    if any(x in q for x in ["summary", "overview", "summarize"]):
        return "summary"

    # Numeric / factual queries
    if any(x in q for x in ["how many", "percentage", "percent", "number", "value", "when", "who"]):
        return "fact"

    return "analysis"



# ---------------------- RETRIEVAL --------------------------
def retrieve_context(
    query: str,
    paper_id: str,
    top_k: int = 8,
    initial_k: int = 60
) -> List[Dict]:

    q_emb = embedder.encode([query], convert_to_numpy=True)[0]

    res = collection.query(
        query_embeddings=[q_emb.tolist()],
        n_results=initial_k
    )

    docs = []
    for doc, meta in zip(res["documents"][0], res["metadatas"][0]):
        if meta.get("paper_id") == paper_id:
            docs.append({"doc": doc, "meta": meta})

    if not docs:
        return []

    scores = reranker.predict([[query, d["doc"]] for d in docs])
    idxs = np.argsort(scores)[::-1][:top_k]

    return [docs[i] for i in idxs]


# ---------------------- PROMPT -----------------------------
def build_citation_prompt(context_docs, query, mode):

    context = "\n\n".join(
        [f"[{i+1}] {d['doc'][:1200]}" for i, d in enumerate(context_docs)]
    )

    if mode == "fact":
        return f"""
Answer the question using ONLY the context below.
Write ONE factual sentence.
Every claim MUST end with a citation like [1], [2].

If the answer is not supported by the context, respond exactly:
"Not mentioned in the provided context."

Context:
{context}

Question: {query}

Answer:
""".strip()

    if mode == "summary":
        return f"""
Summarize the paper using ONLY the context below.
Write 5–7 sentences.
Every sentence MUST include at least one citation [1], [2].
Do NOT add external knowledge.

Context:
{context}

Answer:
""".strip()

    # analysis
    return f"""
You are an academic research assistant.

Answer using ONLY the context below.
Write EXACTLY 4 numbered sections:
1. Key Problem
2. Proposed Method
3. Experimental Evidence
4. Limitations

Each section must contain 2–3 sentences.
Every sentence MUST include citations like [1], [2].
If a section cannot be supported, write:
"Not mentioned in the provided context."

Context:
{context}

Answer:
""".strip()

def build_qa_prompt(context_docs, query):
    context = "\n\n".join(
        [f"[{i+1}] {d['doc'][:1200]}" for i, d in enumerate(context_docs)]
    )

    return f"""
Answer the question using ONLY the context below.

Answer in ONE sentence describing ONLY the core problem addressed by the paper.
Do NOT describe methods, solutions, results, or future work.

If the answer is not present, respond exactly:
"Not mentioned in the provided context."

Context:
{context}

Question: {query}

Answer:
""".strip()




# ---------------------- GENERATION -------------------------
def generate_answer(prompt, max_new_tokens=450):
    messages = [{"role": "user", "content": prompt}]

    out = generator(
        messages,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    generated = out[0]["generated_text"]

    # Case 1: Chat-style output (Qwen, LLaMA chat)
    if isinstance(generated, list):
        for msg in reversed(generated):
            if msg.get("role") == "assistant":
                return msg.get("content", "").strip()
        return ""

    # Case 2: Plain text output
    if isinstance(generated, str):
        return generated.strip()

    # Fallback
    return str(generated).strip()





# ---------------------- MAIN RAG ---------------------------
def rag_answer(query: str, paper_id: str):

    mode = detect_mode(query)

    context_docs = retrieve_context(query, paper_id)

    if not context_docs:
        return {
            "answer": "No relevant context found in the paper.",
            "context": []
        }

    print("\n===== RETRIEVED CONTEXT =====")
    for i, d in enumerate(context_docs):
        print(f"\n--- Chunk [{i+1}] ---")
        print(d["doc"][:400])

    if mode == "qa":
      prompt = build_qa_prompt(context_docs, query)
    else:
      prompt = build_citation_prompt(context_docs, query, mode)

    answer = generate_answer(prompt)


    return {
        "answer": answer,
        "context": context_docs
    }


In [ ]:
rag_answer(
    query="What is the main research focus of the paper?",
    paper_id="1803.06161v2"
)



===== RETRIEVED CONTEXT =====

--- Chunk [1] ---
the distribution of freely available documents by web domains? As other studies had previously hinted (Jamali & Nabavi, 2015; Martín-Martín et al., 2014), the main source of freely available documents according to Google Scholar is, by far, the academic social network ResearchGate, which provided free access to 32.6% of all the documents in our sample (almost the same amount as all publishers and 

--- Chunk [2] ---
were freely accessible Laakso & Polonioli, 2018 Publication lists of ethics researchers Ethics 2010- 2015 Articles 1,682 Google Scholar 2017 Publication years, OA types 56% of the documents were freely accessible 12 1.6. Research questions This paper mainly intends to ascertain the suitability of the data available in Google Scholar to gauge the levels of adoption of OA in scientific journal artic

--- Chunk [3] ---
institutions, funders and publishers to OA In the beginnings of the OA movement, a great emphasis was put on t

{'answer': 'The paper aims to determine how much of recently published scientific literature is freely available according to data available in Google Scholar, by year of publication, subject categories, and country of affiliation of the authors.',
 'context': [{'doc': 'the distribution of freely available documents by web domains? As other studies had previously hinted (Jamali & Nabavi, 2015; Martín-Martín et al., 2014), the main source of freely available documents according to Google Scholar is, by far, the academic social network ResearchGate, which provided free access to 32.6% of all the documents in our sample (almost the same amount as all publishers and repositories put together). ResearchGate has a strong presence in Google Scholar, demonstrated by the fact that Google Scholar selects the ResearchGate version of an article as the primary version (see Figure 2) in 43.8% of the cases. After ResearchGate, among the first places of the rank of websites that provided more freely \

In [ ]:
# =========================
# LIVE PDF INGESTION UTILS
# =========================

import fitz
import tempfile
import uuid
from nltk.tokenize import sent_tokenize

def ingest_uploaded_pdf(pdf_bytes, filename):
    # Create temp file
    tmp_path = f"/tmp/{uuid.uuid4()}_{filename}"
    with open(tmp_path, "wb") as f:
        f.write(pdf_bytes)

    # Extract text
    doc = fitz.open(tmp_path)
    text = "\n".join([p.get_text() for p in doc])
    if len(text.strip()) < 200:
        return None, "PDF text extraction failed."

    # Chunking
    sentences = sent_tokenize(text)
    chunks = chunk_sentences(sentences, CHUNK_SIZE, CHUNK_OVERLAP)

    paper_id = filename.replace(".pdf", "")

    ids = [f"{paper_id}__upload__{i:04d}" for i in range(len(chunks))]
    metas = [{"paper_id": paper_id, "section": "uploaded", "chunk_id": i}
             for i in range(len(chunks))]

    for i in range(0, len(chunks), BATCH_SIZE):
        docs = chunks[i:i+BATCH_SIZE]
        embs = embedder.encode(docs, convert_to_numpy=True)
        collection.add(
            ids=ids[i:i+BATCH_SIZE],
            documents=docs,
            embeddings=embs.tolist(),
            metadatas=metas[i:i+BATCH_SIZE]
        )

    return paper_id, f"Indexed {len(chunks)} chunks successfully."


In [ ]:
# ==========================================================
# CELL I — Evaluation Set
# ==========================================================

EVAL_SET = [
    {
        "paper_id": "2212.07286v2",
        "question": "What problem does this paper address?",
        "expected_contains": ["accessibility", "disabilities"]
    },
    {
        "paper_id": "2212.07286v2",
        "question": "Which format does the paper propose to improve accessibility?",
        "expected_contains": ["html"]
    },
    {
        "paper_id": "2212.07286v2",
        "question": "What percentage of PDFs are accessible according to the paper?",
        "expected_answer": "Not mentioned in the provided context."
    },
    {
        "paper_id": "2212.07286v2",
        "question": "What approach does arXiv propose to improve accessibility?",
        "expected_contains": ["html", "pdf"]
    },
    {
        "paper_id": "2212.07286v2",
        "question": "Summarize the paper."
    },
    {
        "paper_id": "2212.07286v2",
        "question": "Does the paper discuss blockchain technology?",
        "expected_answer": "Not mentioned in the provided context."
    },
    {
        "paper_id": "1803.06161v2",
        "question": "Does the paper propose a deep learning architecture?",
        "expected_answer": "Not mentioned in the provided context."
    }
]


In [ ]:
# ==========================================================
# CELL J — Evaluation Runner
# ==========================================================

def run_evaluation(eval_set):
    passed = 0

    for i, e in enumerate(eval_set, 1):
        print(f"\n--- Q{i} ---")
        print("Question:", e["question"])

        out = rag_answer(e["question"], e["paper_id"])
        ans = out["answer"]

        print("Answer:", ans)

        if "expected_answer" in e:
            ok = ans.strip() == e["expected_answer"]
        elif "expected_contains" in e:
            ok = all(x.lower() in ans.lower() for x in e["expected_contains"])
        else:
            ok = True  # summaries/manual checks

        print("Result:", "PASS ✅" if ok else "FAIL ❌")
        passed += int(ok)

    print(f"\n🎯 FINAL SCORE: {passed}/{len(eval_set)}")


In [ ]:
run_evaluation(EVAL_SET)



--- Q1 ---
Question: What problem does this paper address?

===== RETRIEVED CONTEXT =====

--- Chunk [1] ---
The research content hosted by arXiv is not fully accessible to everyone
due to disabilities and other barriers. This matters because a significant
proportion of people have reading and visual disabilities, it is important to
our community that arXiv is as open as possible, and if science is to advance,
we need wide and diverse participation. In addition, we have mandates to
become accessible, and

--- Chunk [2] ---
every technical document will just work [with screen readers]. In a smaller dream world, it just works on arXiv.” “My area in physics is stuck in its ways. Not many people want to go against the norm. And arXiv sets the norm.” Because arXiv has direct control over the submission pipeline, it opens up opportunities: “The good news is that the potential for significant impact is just right there. Th

--- Chunk [3] ---
Improving access to research is a broad and inclus

In [ ]:
def run_advanced_evaluation(EVAL_SET):
    total = len(EVAL_SET)
    answered = 0
    abstained = 0
    correct_abstentions = 0
    expected_abstentions = 0
    context_hits = 0
    total_keywords = 0

    for e in EVAL_SET:
        out = rag_answer(e["question"], e["paper_id"])
        ans = out["answer"].lower()
        contexts = " ".join([c["doc"].lower() for c in out["context"]])

        is_abstained = "not mentioned in the provided context" in ans

        if is_abstained:
            abstained += 1
        else:
            answered += 1

        # Abstention accuracy
        if "expected_answer" in e and "not mentioned" in e["expected_answer"].lower():
            expected_abstentions += 1
            if is_abstained:
                correct_abstentions += 1

        # Context grounding
        if "expected_contains" in e:
            for kw in e["expected_contains"]:
                total_keywords += 1
                if kw.lower() in ans and kw.lower() in contexts:
                    context_hits += 1

    print("\n📊 ADVANCED EVALUATION METRICS")
    print("=" * 40)
    print(f"Total questions        : {total}")
    print(f"Answered               : {answered}")
    print(f"Abstained              : {abstained}")
    print(f"Answer rate            : {answered/total:.2f}")
    print(f"Abstention rate        : {abstained/total:.2f}")

    if expected_abstentions > 0:
        print(f"Abstention accuracy    : {correct_abstentions/expected_abstentions:.2f}")

    if total_keywords > 0:
        print(f"Context grounding rate : {context_hits/total_keywords:.2f}")


In [ ]:
run_advanced_evaluation(EVAL_SET)



===== RETRIEVED CONTEXT =====

--- Chunk [1] ---
The research content hosted by arXiv is not fully accessible to everyone
due to disabilities and other barriers. This matters because a significant
proportion of people have reading and visual disabilities, it is important to
our community that arXiv is as open as possible, and if science is to advance,
we need wide and diverse participation. In addition, we have mandates to
become accessible, and

--- Chunk [2] ---
every technical document will just work [with screen readers]. In a smaller dream world, it just works on arXiv.” “My area in physics is stuck in its ways. Not many people want to go against the norm. And arXiv sets the norm.” Because arXiv has direct control over the submission pipeline, it opens up opportunities: “The good news is that the potential for significant impact is just right there. Th

--- Chunk [3] ---
Improving access to research is a broad and inclusive effort, championed and moved forward by
individuals and 

In [ ]:
# =========================
# CELL UI (Gradio)
# =========================
!pip install -q gradio


In [ ]:
import gradio as gr

def ask_question(query, paper_id):
    if not query.strip():
        return "Please enter a question.", ""

    out = rag_answer(query, paper_id)

    evidence = ""
    for i, c in enumerate(out["context"], 1):
        evidence += f"\n--- Chunk {i} ---\n{c['doc'][:800]}\n"

    return out["answer"], evidence


def upload_pdf(file):
    if file is None:
        return "No file uploaded.", []

    paper_id, msg = ingest_uploaded_pdf(file.read(), file.name)
    updated_ids = sorted({m["paper_id"] for m in collection.get()["metadatas"]})

    return msg, updated_ids


paper_ids = sorted({m["paper_id"] for m in collection.get()["metadatas"]})

with gr.Blocks(title="📚 Research Assistant (RAG)") as demo:
    gr.Markdown("## 📚 Research Paper Question Answering System")
    gr.Markdown("Grounded, citation-based answers with zero hallucination.")

    with gr.Tabs():

        # ---------------- ASK TAB ----------------
        with gr.Tab("🔍 Ask Question"):
            paper_dd = gr.Dropdown(paper_ids, label="Select Paper")
            query = gr.Textbox(lines=2, placeholder="Ask a question...")
            ask_btn = gr.Button("Ask")

            answer = gr.Textbox(label="Answer", lines=6)
            evidence = gr.Textbox(label="Retrieved Evidence", lines=12)

            ask_btn.click(ask_question,
                          inputs=[query, paper_dd],
                          outputs=[answer, evidence])

        # ---------------- UPLOAD TAB ----------------
        with gr.Tab("📄 Upload New Paper"):
            upload = gr.File(file_types=[".pdf"])
            upload_btn = gr.Button("Upload & Index")

            status = gr.Textbox(label="Status")
            updated_dd = gr.Dropdown(label="Available Papers")

            upload_btn.click(upload_pdf,
                             inputs=[upload],
                             outputs=[status, updated_dd])

        # ---------------- EVALUATION TAB ----------------
        with gr.Tab("📊 Evaluation"):
            gr.Markdown("""
            **Evaluation Metrics**
            - Context Grounding Rate
            - Abstention Accuracy
            - Answer Coverage
            - Manual QA verification
            """)

        # ---------------- ABOUT TAB ----------------
        with gr.Tab("ℹ️ About"):
            gr.Markdown("""
            **Architecture**
            - SentenceTransformers embeddings
            - Chroma vector DB
            - Cross-encoder reranker
            - Citation-constrained generation
            - Gradio UI (Colab-compatible)

            **Guarantees**
            - No hallucination
            - Evidence-only answers
            - Safe abstention
            """)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://301db488ccfba1de45.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
